# Gold : `fact_orders`

**Entrées :** `silver.orders`, `silver.customers`, `silver.order_items`, `silver.order_reviews`
**Sortie :** `workspace.gold.fact_orders` (**une ligne par commande**, clé : `order_id`)

## Ce qui change de Silver vers Gold
| Modification | Détail |
|---|---|
| **Agrégation avant jointure** | Les articles sont ramenés à une ligne par commande (`item_count`, `product_revenue`, `freight_total`, `order_total`) **avant** de joindre : c'est ce qui évite le piège de la duplication des lignes |
| **Un avis par commande** | On garde l'avis le plus récent (réponse, puis création, puis `review_id`). `review_count` indique combien d'avis la commande avait. Une note invalide devient `NULL` |
| **Localisation de livraison** | Code postal, ville et État de la commande, depuis `silver.customers` |
| **Calculs métier** | `delivery_days`, `delay_vs_estimate_days`, `is_late`, `is_sale`, `order_month` |
| **Utilisation des indicateurs** | Les indicateurs Silver servent de règles de calcul (le délai n'est calculé que pour une commande livrée avec une date de livraison) ; `has_date_anomaly` résume les anomalies de dates |
| **Retrait** | Les colonnes techniques (`_bronze_loaded_at`, `_silver_loaded_at`) |

## Définitions retenues (provisoires, à valider)
| Terme | Définition |
|---|---|
| Vente (`is_sale`) | Commande ni `canceled` ni `unavailable` |
| Délai (`delivery_days`) | Livraison au client − achat, en jours, pour les commandes `delivered` avec une date de livraison |
| Retard (`is_late`) | **Jour** de livraison > **jour** estimé (la date estimée n'a pas d'heure) |
| Écart (`delay_vs_estimate_days`) | Jour de livraison − jour estimé (négatif = en avance) |
| Total (`order_total`) | `SUM(price + freight_value)` par commande |
| Avis retenu | Le plus récent de la commande |

In [0]:
CREATE OR REPLACE TABLE workspace.gold.fact_orders AS
WITH orders_silver AS (
    SELECT
        order_id, customer_id, customer_unique_id, order_status,
        order_purchase_ts   AS order_purchase_timestamp,
        order_approved_ts   AS order_approved_at,
        order_carrier_ts    AS order_delivered_carrier_date,
        order_delivered_ts  AS order_delivered_customer_date,
        order_estimated_ts  AS order_estimated_delivery_date,
        flag_r01_delivered_no_delivery_date AS flag_delivered_sans_livraison,
        flag_r02_delivered_no_approval      AS flag_delivered_sans_approbation,
        flag_r03_delivered_no_carrier       AS flag_delivered_sans_transporteur,
        flag_r04_canceled_with_delivery     AS flag_canceled_avec_livraison,
        flag_r05_carrier_before_purchase    AS flag_transporteur_avant_achat,
        flag_r06_carrier_before_approval    AS flag_transporteur_avant_approbation,
        flag_r08_delivery_before_approval   AS flag_livraison_avant_approbation,
        flag_r09_delivery_before_carrier    AS flag_livraison_avant_transporteur
    FROM workspace.silver.silver_orders
),
items AS (
    SELECT
        order_id,
        COUNT(*)                     AS item_count,
        COUNT(DISTINCT product_id)   AS distinct_product_count,
        SUM(price)                   AS product_revenue,
        SUM(freight_value)           AS freight_total,
        SUM(price + freight_value)   AS order_total
    FROM workspace.silver.silver_order_items
    GROUP BY order_id
),
reviews AS (
    SELECT order_id, review_score, review_count
    FROM (
        SELECT
            order_id,
            CASE WHEN review_score BETWEEN 1 AND 5 THEN review_score END AS review_score,
            COUNT(*) OVER (PARTITION BY order_id) AS review_count,
            ROW_NUMBER() OVER (
                PARTITION BY order_id
                ORDER BY review_answer_ts DESC, review_creation_ts DESC, review_id DESC
            ) AS rn
        FROM workspace.silver.silver_order_reviews
    )
    WHERE rn = 1
),
base AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.customer_unique_id,
        o.order_status,
        (o.order_status NOT IN ('canceled', 'unavailable'))        AS is_sale,
        CAST(o.order_purchase_timestamp AS DATE)                   AS order_date,
        DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM')         AS order_month,
        o.order_purchase_timestamp,
        o.order_approved_at,
        o.order_delivered_carrier_date,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,
        c.customer_zip_code_prefix                                 AS delivery_zip_prefix,
        c.customer_city                                            AS delivery_city,
        c.customer_state                                           AS delivery_state,
        COALESCE(i.item_count, 0)                                  AS item_count,
        i.distinct_product_count,
        COALESCE(i.product_revenue, 0)                             AS product_revenue,
        COALESCE(i.freight_total, 0)                               AS freight_total,
        COALESCE(i.order_total, 0)                                 AS order_total,
        r.review_score,
        COALESCE(r.review_count, 0)                                AS review_count,
        CASE WHEN o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
             THEN ROUND((unix_timestamp(o.order_delivered_customer_date)
                        - unix_timestamp(o.order_purchase_timestamp)) / 86400.0, 1)
        END                                                        AS delivery_days,
        CASE WHEN o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
             THEN DATEDIFF(CAST(o.order_delivered_customer_date AS DATE),
                           CAST(o.order_estimated_delivery_date AS DATE))
        END                                                        AS delay_vs_estimate_days,
        (o.flag_delivered_sans_livraison OR o.flag_delivered_sans_approbation
         OR o.flag_delivered_sans_transporteur OR o.flag_canceled_avec_livraison
         OR o.flag_transporteur_avant_achat OR o.flag_transporteur_avant_approbation
         OR o.flag_livraison_avant_approbation OR o.flag_livraison_avant_transporteur
        )                                                          AS has_date_anomaly
    FROM orders_silver o
    LEFT JOIN workspace.silver.silver_customers c ON o.customer_id = c.customer_id
    LEFT JOIN items i   ON o.order_id = i.order_id
    LEFT JOIN reviews r ON o.order_id = r.order_id
)
SELECT
    *,
    (delay_vs_estimate_days > 0) AS is_late
FROM base

num_affected_rows,num_inserted_rows


## Vérification 1 : aucune duplication, montants exacts
| Mesure | Attendu |
|---|---:|
| `lignes`, `commandes_distinctes` | 99 441 |
| `articles` | 112 650 |
| `commandes_avec_articles` | 98 666 |
| `ca_produits` | **13 591 643,70** (le chiffre d'affaires de l'exploration) |
| `avec_avis` | proche de 98 673 (commandes ayant au moins un avis) |

In [0]:
SELECT
    COUNT(*) AS lignes,
    COUNT(DISTINCT order_id) AS commandes_distinctes,
    SUM(item_count) AS articles,
    COUNT(*) FILTER (WHERE item_count > 0) AS commandes_avec_articles,
    SUM(product_revenue) AS ca_produits,
    SUM(freight_total) AS port_total,
    SUM(order_total) AS total_articles_et_port,
    COUNT(*) FILTER (WHERE review_count > 0) AS avec_avis,
    COUNT(*) FILTER (WHERE review_count > 1) AS avec_plusieurs_avis
FROM workspace.gold.fact_orders

lignes,commandes_distinctes,articles,commandes_avec_articles,ca_produits,port_total,total_articles_et_port,avec_avis,avec_plusieurs_avis
99441,99441,112650,98666,13591643.70,2251909.54,15843553.24,98673,547


## Vérification 2 : délai et retard
Le nombre de commandes avec `delivery_days` doit correspondre aux commandes `delivered` ayant une date de livraison (96 478 − 8 = **96 470**). Compare `en_retard` avec le résultat de la cellule « jour contre instant » de l'exploration (`en_retard_date`).

In [0]:
SELECT
    COUNT(*) FILTER (WHERE delivery_days IS NOT NULL) AS avec_delai,
    COUNT(*) FILTER (WHERE is_late) AS en_retard,
    COUNT(*) FILTER (WHERE is_late = FALSE) AS pas_en_retard,
    ROUND(AVG(delivery_days), 1) AS delai_moyen_jours,
    MIN(delay_vs_estimate_days) AS ecart_min,
    MAX(delay_vs_estimate_days) AS ecart_max,
    COUNT(*) FILTER (WHERE has_date_anomaly) AS commandes_avec_anomalie_de_dates
FROM workspace.gold.fact_orders

avec_delai,en_retard,pas_en_retard,delai_moyen_jours,ecart_min,ecart_max,commandes_avec_anomalie_de_dates
96470,6534,89936,12.6,-147,188,1411


## Vérification 3 : la jointure aurait-elle gonflé le chiffre d'affaires ?
Cette requête rejoue le piège : on joint **directement** les articles aux commandes, sans agréger. Le total doit être égal à `ca_produits` : sinon, la table Gold est fausse.

In [0]:
SELECT
    (SELECT SUM(product_revenue) FROM workspace.gold.fact_orders) AS ca_gold,
    (SELECT SUM(price) FROM workspace.silver.silver_order_items)         AS ca_silver,
    (SELECT SUM(product_revenue) FROM workspace.gold.fact_orders)
      = (SELECT SUM(price) FROM workspace.silver.silver_order_items)     AS identiques

ca_gold,ca_silver,identiques
13591643.70,13591643.70,true
